# DailyConway — Headless GPU Miner (Colab)

> **Before running:** Go to **Runtime → Change runtime type** and select **T4 GPU**.

This notebook:
1. Checks the GPU and CUDA version
2. Installs all required packages
3. Clones the DailyConway repository
4. (Optionally) mounts Google Drive to persist hits
5. Runs the headless miner with your config

## 1 — Check GPU

In [ ]:
import subprocess, re

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or result.stderr)

nvcc = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
cuda_ver_str = nvcc.stdout
print(cuda_ver_str)

m = re.search(r'release (\d+)\.', cuda_ver_str)
CUDA_MAJOR = int(m.group(1)) if m else 12
print(f'Detected CUDA major version: {CUDA_MAJOR}')

## 2 — Install dependencies

In [ ]:
# Pick the right CuPy wheel for the installed CUDA version
cupy_pkg = f'cupy-cuda{CUDA_MAJOR}x'
print(f'Installing {cupy_pkg} ...')

!pip install -q {cupy_pkg} numba numpy

# Verify CuPy sees the GPU
import cupy as cp
print('CuPy version:', cp.__version__)
print('GPU:', cp.cuda.runtime.getDeviceProperties(0)['name'].decode())

## 3 — Clone the repository

In [ ]:
import os

REPO_URL = 'https://github.com/TheKerbecs/DailyConway.git'
REPO_DIR = '/content/DailyConway'

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

## 4 — (Optional) Mount Google Drive to persist hits

Skip this cell if you just want to download hits at the end of the session.

In [ ]:
USE_DRIVE = True  # set False to save locally inside Colab only

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/DailyConway/hits'
else:
    OUTPUT_DIR = '/content/DailyConway/hits'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Hits will be saved to:', OUTPUT_DIR)

## 5 — Configure the miner

Edit the fields below to match your search criteria.

**`groups`** — each inner list is an AND-group; groups are OR-ed together.  
- `"front"` — the hash must start with `value`  
- `"back"` — the hash must end with `value`  
- `"any"` — the hash must contain `value` anywhere

Alternatively you can use a **`regex`** dict instead of `groups` (see README).

In [ ]:
CONFIG = {
    # ── Identity ──────────────────────────────────────────────────────────
    "owner": "Bobinou",           # your handle, embedded in every hit file

    # ── GPU tuning ────────────────────────────────────────────────────────
    # Colab T4 has 2560 CUDA cores. Good starting point: blocks=512, threads=256.
    # Increase blocks for higher throughput (watch for OOM errors).
    "blocks":   512,
    "threads":  256,
    "ipt":      256,   # iterations per thread per kernel launch
    "workers":  2,     # CPU worker threads post-processing hits

    # ── Search criteria ───────────────────────────────────────────────────
    # Same format as headless.example.json
    "groups": [
        [{"position": "front", "value": "513"}],
        [{"position": "any",   "value": "b0b1400"},
         {"position": "any",   "value": "626f62696e6f75"}]
    ],

    # ── Output ────────────────────────────────────────────────────────────
    "output_dir": OUTPUT_DIR,
    "no_rle": False,   # set True to skip RLE generation (faster saves)
    "quiet":  False,   # set True to suppress per-tick log lines
}

import json
print(json.dumps(CONFIG, indent=2))

## 6 — Run the miner

The cell below runs until you **interrupt the kernel** (Runtime → Interrupt) or the Colab session times out.  
Hit files accumulate in `OUTPUT_DIR` while it runs.

In [ ]:
import sys, os

# Make sure the repo root is on the path so `from app.xxx import ...` works
repo_root = '/content/DailyConway'
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.chdir(repo_root)

import asyncio
from app.headless import _amain

try:
    asyncio.run(_amain(CONFIG))
except KeyboardInterrupt:
    print('\n[stopped] Mining interrupted.')

## 7 — Inspect & download results

Run the cells below **after** stopping the miner.

In [ ]:
import json, os

hits = []
for fname in os.listdir(OUTPUT_DIR):
    if not fname.endswith('.json'):
        continue
    with open(os.path.join(OUTPUT_DIR, fname)) as f:
        d = json.load(f)
    hits.append({'file': fname, 'iterations': d.get('iterations', 0), 'peak': d.get('peak', 0)})

hits.sort(key=lambda h: h['iterations'], reverse=True)
print(f"{'Rank':<6} {'Iterations':<12} {'Peak':<6} File")
print('-' * 70)
for i, h in enumerate(hits[:20], 1):
    print(f"{i:<6} {h['iterations']:<12} {h['peak']:<6} {h['file']}")
print(f'\nTotal hits: {len(hits)}')

In [ ]:
# Download all hits as a zip (only useful when NOT using Drive)
import shutil
from google.colab import files

zip_path = '/content/hits.zip'
shutil.make_archive('/content/hits', 'zip', OUTPUT_DIR)
files.download(zip_path)
print('Download started.')